<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/sweep_ckpt_circuit_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install transformer_lens

In [40]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig, utils
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [41]:
from huggingface_hub import hf_hub_download

REPO_ID = "sojup/entity_binding_test"
FILENAME = "D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [42]:
REPO_ID = "sojup/entity_binding_test"
FILENAME = "id_to_entity.csv"

id_mapping_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [43]:
E = 100
T = 10
D_VOCAB = E + T + 3

In [44]:
N_LAYERS = 3
HEADS = 2

d_model = 256
n_ctx   = 64

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

Moving model to device:  cpu
Model loaded successfully.


In [45]:
id_mapping_df = pd.read_csv(id_mapping_path)
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [46]:
id_to_entity[100] = 'loves'
id_to_entity[101] = 'works with'
id_to_entity[102] = 'interacts with'
id_to_entity[103] = 'lives with'
id_to_entity[104] = 'has a grudge against'
id_to_entity[105] = 'is interested in'
id_to_entity[106] = 'plays with'
id_to_entity[107] = 'goes to school with'
id_to_entity[108] = 'is jealous of'
id_to_entity[109] = 'wants'


In [47]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [48]:
from datasets import load_dataset

dataset = load_dataset("sojup/entity_binding", split="test")

In [49]:
test_df = dataset.to_pandas()
test_dataset = EntityBindingDataset(test_df)

In [50]:
example, label = test_dataset[1]
example, label, [id_to_entity[i.item()] for i in example], id_to_entity[label.item()]

(tensor([ 55, 104,  78, 110,  62, 108,  43, 110,  52, 101,  81, 110,   1, 107,
          68, 110,  45, 105,  59, 110,  18, 106,  78, 110,  50, 109,  43, 110,
         108,  62, 111]),
 tensor(43),
 ['Chelsea',
  'has a grudge against',
  'Joseph',
  ',',
  'Angela',
  'is jealous of',
  'Keith',
  ',',
  'Joseph',
  'works with',
  'Jose',
  ',',
  'Angel',
  'goes to school with',
  'Leslie',
  ',',
  'Jesse',
  'is interested in',
  'Noah',
  ',',
  'Lisa',
  'plays with',
  'Joseph',
  ',',
  'Mitchell',
  'wants',
  'Keith',
  ',',
  'is jealous of',
  'Angela',
  '?'],
 'Keith')

In [51]:
## considering the incorrect token to be the 10th most probable prediction
## take a batch of examples with this heuristic

In [52]:
try:
    import google.colab # type: ignore
    IN_COLAB = True
except:
    IN_COLAB = False

import os, sys
chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"

if IN_COLAB:
    # Install packages
    %pip install einops
    %pip install jaxtyping
    %pip install git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

    # Code to download the necessary files (e.g. solutions, test funcs)
    if not os.path.exists(f"/content/{chapter}"):
        !wget https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/main.zip
        !unzip /content/main.zip 'ARENA_3.0-main/chapter1_transformer_interp/exercises/*'
        sys.path.append(f"/content/{repo}-main/{chapter}/exercises")
        os.remove("/content/main.zip")
        os.rename(f"{repo}-main/{chapter}", chapter)
        os.rmdir(f"{repo}-main")
        os.chdir(f"{chapter}/exercises")
else:
    chapter_dir = r"./" if chapter in os.listdir() else os.getcwd().split(chapter)[0]
    sys.path.append(chapter_dir + f"{chapter}/exercises")


  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-dtjm93hz
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-dtjm93hz
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [53]:
### Attention Pattern

In [54]:
logits, cache = model.run_with_cache(example, remove_batch_dim=True)

In [55]:
probs = torch.softmax(logits[0, -1, :], dim=-1)

In [56]:
for i, idx in enumerate(torch.topk(probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

Keith 0.9996976852416992
Michelle 0.0002580875006970018
Jeffery 2.109161505359225e-05
Joseph 2.5333204121125164e-06
Susan 2.0856975879723905e-06


Correct label: Keith


In [57]:
### Taking Michelle as corrupt ex.

In [58]:
### Method 1: Residual stream patching

In [59]:
example

tensor([ 55, 104,  78, 110,  62, 108,  43, 110,  52, 101,  81, 110,   1, 107,
         68, 110,  45, 105,  59, 110,  18, 106,  78, 110,  50, 109,  43, 110,
        108,  62, 111])

In [60]:
id_to_entity_rev = {v: k for k, v in id_to_entity.items()}

In [61]:
id_to_entity_rev["Michelle"], id_to_entity_rev["Keith"]

(99, 43)

In [62]:
corrupt_example = example.clone()
corrupt_example[corrupt_example == id_to_entity_rev["Keith"]] = id_to_entity_rev["Michelle"]

In [63]:
corrupt_example

tensor([ 55, 104,  78, 110,  62, 108,  99, 110,  52, 101,  81, 110,   1, 107,
         68, 110,  45, 105,  59, 110,  18, 106,  78, 110,  50, 109,  99, 110,
        108,  62, 111])

In [64]:
corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_example)

In [65]:
corrupt_probs = torch.softmax(corrupt_logits[0, -1, :], dim=-1)

In [66]:
for i, idx in enumerate(torch.topk(corrupt_probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(corrupt_probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

Michelle 0.9999959468841553
Chelsea 2.2281610654317774e-06
Allison 4.950758238919661e-07
Erica 3.1213855322675954e-07
Robert 2.588238032785739e-07


Correct label: Keith


In [67]:
def patch_residual_stream(activations, hook, layer="blocks.6.hook_resid_post", pos=5):
   activations[:, pos, :] = corrupt_cache[layer][:, pos, :]
   return activations

In [68]:
import torch
from functools import partial

layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]
n_layers = len(layers)
n_pos = len(example)

clean_answer_index = 43
corrupt_answer_index = 99

# Test the effect of patching at any layer and any position
patching_effect = torch.zeros(n_layers, n_pos)
for l, layer in enumerate(layers):
    for pos in range(n_pos):
        fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
        prediction_logits = model.run_with_hooks(example,
                                                 fwd_hooks=fwd_hooks)[0, -1]
        patching_effect[l, pos] = prediction_logits[clean_answer_index] \
                                  - prediction_logits[corrupt_answer_index]

In [80]:
import plotly.express as px
import torch
from functools import partial

def imshow(
    tensor,
    xlabel="X",
    ylabel="Y",
    zlabel=None,
    xticks=None,
    yticks=None,
    c_midpoint=0.0,
    c_scale="RdBu",
    show=True,
    **kwargs
):
    tensor = utils.to_numpy(tensor)
    n_rows, n_cols = tensor.shape

    labels = {"x": xlabel, "y": ylabel}
    if zlabel is not None:
        labels["color"] = zlabel

    # Build the figure with numeric axes
    fig = px.imshow(
        tensor,
        labels=labels,
        color_continuous_midpoint=c_midpoint,
        color_continuous_scale=c_scale,
        **kwargs
    )

    # Map numeric positions -> your (possibly duplicate) tokens
    if xticks is not None:
        xtxt = [str(x) for x in xticks]
        fig.update_xaxes(
            tickmode="array",
            tickvals=list(range(n_cols)),
            ticktext=xtxt,
            type="linear",   # ensure numeric axis, not categorical
            tickangle=-90 # Rotate x-axis labels vertically
        )

    if yticks is not None:
        ytxt = [str(y) for y in yticks]
        fig.update_yaxes(
            tickmode="array",
            tickvals=list(range(n_rows)),
            ticktext=ytxt,
            type="linear"
        )

    return fig

In [78]:
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layers, xlabel="pos", ylabel="layer",
       zlabel="Logit difference", title="Patching with other name", width=800, height=380)

In [71]:
## Hypothesis: Information is stored in Layer 0 at the Keith token (maybe current token attention) - then the information travels to the comma?
## And then just at the last layer, it travels to the final token

In [72]:
def patch_head_result(activations, hook, layer=None, head=None, pos=None):
   activations[:, pos, head, :] = corrupt_cache[hook.name][:, pos, head, :]
   return activations

In [81]:
n_layers = 3
n_heads = 2
n_pos = 19

_, corrupt_cache = model.run_with_cache(corrupt_example)


patching_effect = torch.zeros(n_layers*n_heads, n_pos)
for layer in range(n_layers):
    for head in range(n_heads):
        for pos in range(n_pos):
            fwd_hooks = [(
                f"blocks.{layer}.attn.hook_result",
                    partial(patch_head_result, layer=layer, head=head, pos=pos)
            )]
            prediction_logits = model.run_with_hooks(example,
                                                     fwd_hooks=fwd_hooks)[0, -1]
            patching_effect[n_heads*layer+head, pos] =  \
                                    prediction_logits[clean_answer_index] \
                                    - prediction_logits[corrupt_answer_index]


token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(example)]
layerhead_labels = [f"{l}.{h}" for l in range(n_layers) for h in range(n_heads)]
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layerhead_labels, xlabel="position", ylabel="layer.head",
           zlabel="Logit difference", title=f"Patching with Michelle instead of Keith", width=700, height=800, tickangle=-90)

In [74]:
### Attention Pattern

In [75]:
import circuitsvis as cv
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

def tensor_to_numpy(t):
    if isinstance(t, torch.Tensor):
        t = t.detach().cpu().numpy()
    return t

str_tokens = [id_to_entity[i.item()] for i in example]
for layer in range(model.cfg.n_layers):
    attention_pattern = cache["pattern", layer]
    display(Markdown(f"### Layer {layer}"))
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern))

### Layer 0

### Layer 1

### Layer 2

In [76]:
cache["pattern", 0].shape

torch.Size([2, 31, 31])